In [1]:
import io
from google.cloud import storage
from nptdms import TdmsFile

In [2]:
# --- GCS parameters ---
BUCKET_NAME = "emotor-dataset-raw"
TDMS_BLOB   = "current,temp/0Nm_BPFI_30.tdms"

# --- Read TDMS from GCS into memory ---
client = storage.Client()
bucket = client.bucket(BUCKET_NAME)
blob = bucket.blob(TDMS_BLOB)

tdms_bytes = blob.download_as_bytes()
tdms = TdmsFile(io.BytesIO(tdms_bytes))

# --- Extract sampling frequency ---
fs = None

for group in tdms.groups():
    if group.name != "Log":
        continue

    for ch in group.channels():
        props = ch.properties

        if "wf_increment" in props:
            dt = props["wf_increment"]
            fs = 1 / dt
            break

    if fs is not None:
        break

if fs is None:
    raise ValueError("Sampling frequency (fs) not found in TDMS metadata")

print(f"Sampling frequency fs = {fs:.2f} Hz")

Sampling frequency fs = 25608.19 Hz
